## NEU 171L Lab 1 - Analysis of MRI data
Jupyter is an interactive, web-based python interface. To run each cell use `ctrl+enter`. Three packages are used here: numpy (computation + sampling), pandas (data + statistics methods), and matplotlib (plotting).

### Learning Objectives
- Use descriptive statistics (mean, standard deviation, standard error of the mean) to quantify the basic features of data
- Use a **bootstrap** to build the sampling distribution of a mean, and report a 95% confidence interval from it
- Use a **permutation test** to ask whether a difference between two groups is larger than expected if the group labels were meaningless
- Understand what a **p-value** means with respect to the tested hypothesis


In [ ]:
# RUN THIS CELL to import python packages
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd


### Step 1: Load in and clean the data

In [ ]:
# Step 1: Load in + clean the data
data = (pd.read_csv("oasis_cross-sectional.csv")
        .dropna(axis=0, subset=["CDR", "nWBV"]))   # drop participants without CDR or nWBV

dementia_data = data.loc[data.CDR > 0, :]          # nonzero clinical dementia rating
healthy_data  = data.loc[data.CDR == 0.0, :]       # general (healthy) population

print("Sample of dementia data:")
print(dementia_data.head(10), "\n")
print("Sample of healthy data:")
print(healthy_data.head(10), "\n")


We are only seeing the first 10 rows in each data set (that is what `.head(10)` does). Just looking at the data, which group seems to have the higher brain volume (nWBV)?

### Step 2: Basic descriptive statistics

In this section you will summarize the normalized whole-brain volume (nWBV) of each group with three numbers: the **mean**, the **standard deviation (SD)**, and the **standard error of the mean (SEM)**.

The SEM measures how precisely we have estimated the *mean* — it shrinks as the sample gets larger:

$$\mathrm{SEM} = \frac{\mathrm{SD}}{\sqrt{N}}$$

where $N$ is the sample size. So before we compute the SEM, we first need $N$ for each group.

**First, the sample size.** Run the cell below to count participants in each group and save them as `n_healthy` and `n_dementia`. `.count()` counts the values in a column.

In [ ]:
# Sample size of each group (already complete -- just run it).
n_healthy   = healthy_data["nWBV"].count()
n_dementia  = dementia_data["nWBV"].count()

print("Sample size (N) of the healthy population:  %d" % n_healthy)
print("Sample size (N) of the dementia population: %d" % n_dementia)


**Next, the mean.** Calculate the **mean** nWBV for the healthy subjects. Apply the `mean` method to the "nWBV" column of `healthy_data` and print it. `.3f` prints three decimal places.

In [ ]:
# Mean of the healthy group (fill in the column name)
mean_healthy = healthy_data[...].mean()
print("Mean nWBV for the healthy population is %.3f" % (mean_healthy))


Calculate the **mean** nWBV for the dementia patients following the same pattern.

In [ ]:
# Mean of the dementia group
mean_dementia = ...
print(...)


**Next, the standard deviation.** Calculate and print the **SD** of the nWBV for the healthy and dementia patients. The SD measures how spread out the individual values are. Method syntax is `.std()`.

In [ ]:
# Standard deviation of each group (fill in using the .std() method)
sd_healthy  = healthy_data[...].std()
sd_dementia = ...

print("SD of nWBV for the healthy population is  %.3f" % (sd_healthy))
print(...)


**Finally, the standard error of the mean.** Combine the SD and sample size to get the SEM for the **healthy** group:

$$\mathrm{SEM} = \frac{\mathrm{SD}}{\sqrt{N}}$$

Syntax you need: `np.sqrt(x)` for the square root, and `/` for division. You already have `sd_healthy` and `n_healthy`.

In [ ]:
# Standard error of the mean for the HEALTHY group.
# SEM = SD / sqrt(N)
# hint: sd_healthy on top, np.sqrt(n_healthy) on the bottom.
sem_healthy = ... / np.sqrt(...)

print("SEM of nWBV for the healthy population is %.4f" % (sem_healthy))


In [ ]:
# Plot the data as a histogram (just run this cell)
fig, ax = plt.subplots(1, figsize=(10,8))
ax.hist(healthy_data["nWBV"].values, rwidth=0.5, alpha=0.75, bins=10, label='healthy_data')
ax.hist(dementia_data["nWBV"].values, rwidth=0.5, alpha=0.75, bins=10, label='dementia_data')
ax.set_title("Distribution of nWBV for Dementia vs Healthy Population")
ax.set_xlabel("nWBV (A.U.)")
ax.set_ylabel("Count")
ax.legend()


### Step 2b: Bootstrapping the sampling distribution of the mean

The SEM you just calculated with the formula estimates how much the *mean* would bounce around if you repeatedly re-sampled the healthy population. A **bootstrap** lets us see that variability directly instead of trusting a formula, and lets us read a **95% confidence interval** straight off the resampled means.

The idea: treat the healthy sample as a stand-in for the whole population and draw many new samples from it **with replacement** (the same participant can be picked more than once). Each resample gives a new mean. Thousands of these means form the **sampling distribution of the mean** — its spread should match the SEM, and its 2.5th–97.5th percentiles give a 95% CI.

**Your task:** each resample must be the **same size as the original sample**. Fill in the `size=` argument with the right value. 

Note: `replace=True` is what makes this bootstrap run **with replacement**

In [ ]:
# Bootstrap the healthy group to build the sampling distribution of its mean.
n_iterations = 10000
bootstrap_means = np.zeros([n_iterations,])

for i in range(n_iterations):

    # How many samples should be chosen for each bootstrap resample? Fill in the correct variable
    n_to_draw_each_resample = ...

    # Draw a bootstrap sample from the healthy group, WITH replacement.
    current_sample = np.random.choice(a=healthy_data["nWBV"],
                                      size=n_to_draw_each_resample,
                                      replace=True)
    bootstrap_means[i] = current_sample.mean()

# The spread of the bootstrap means IS an estimate of the SEM:
print("SEM from the formula (SD / sqrt(N)):       %.4f" % sem_healthy)
print("Std of the bootstrap means (should match): %.4f" % bootstrap_means.std())

# This code calculates 95% confidence interval straight from the bootstrap distribution.
# because np.percentile(bootstrap_means, [2.5, 97.5]) returns the lower and upper edges.
ci_low, ci_high = np.percentile(bootstrap_means, [2.5, 97.5])
print("Bootstrap 95%% CI for the healthy mean nWBV: [%.4f, %.4f]" % (ci_low, ci_high))

# Plot the sampling distribution with the mean, +/- 1 SEM, and the bootstrap 95% CI.
# (no edits needed, this part of the code will just run and produce a figure)
fig, ax = plt.subplots(1, figsize=(10, 8))
ax.hist(bootstrap_means, rwidth=0.9, alpha=0.5, bins=30,
        label="Bootstrap distribution of the healthy mean")
ax.axvline(mean_healthy, color="black", lw=2, label="Healthy mean")
ax.axvline(mean_healthy - sem_healthy, color="C1", lw=2, ls="--", label="- 1 SEM")
ax.axvline(mean_healthy + sem_healthy, color="C1", lw=2, ls="--", label="+ 1 SEM")
ax.axvline(ci_low,  color="C3", lw=2, ls=":", label="2.5% (bootstrap 95% CI)")
ax.axvline(ci_high, color="C3", lw=2, ls=":", label="97.5% (bootstrap 95% CI)")
ax.set_title("Bootstrap sampling distribution of the healthy mean nWBV")
ax.set_xlabel("Mean nWBV (A.U.)")
ax.set_ylabel("Count")
ax.legend()


### Step 3: Permutation test

We want to know whether healthy and dementia participants really differ in nWBV, or whether the difference could easily have arisen by chance.

**Null hypothesis:** the "healthy" and "dementia" labels are arbitrary — every participant's nWBV came from the *same* underlying distribution. If that were true, splitting participants into the two groups would be no different from shuffling the labels and dealing them into two random groups of the same sizes.

A **permutation test** builds the null distribution directly:
1. Compute the **observed statistic** — the real difference in mean nWBV (healthy minus dementia).
2. Pool all participants, **shuffle the labels**, and re-split into random groups the size of the orignal groups.
3. Recompute the difference in means for that shuffled split — this gives you one mean value for the null distribution.
4. Repeat many times (here, 10,000) to build the whole null distribution.

**Why shuffle *without* replacement?** We are re-assigning labels to a fixed set of real participants; each appears exactly once per shuffle, just possibly in the other group. That differs from a **bootstrap**, which samples *with* replacement to imitate collecting brand-new data. `np.random.permutation` reshuffles the pooled values without replacement.

Fill in the sections marked `...`.

In [ ]:
# We shuffle the labels and record the difference in means each time (10,000 shuffles).
n_iterations = 10000

# Pool ALL participants' nWBV values (healthy + dementia) into one array.
pooled_nWBV = pd.concat([healthy_data.nWBV, dementia_data.nWBV]).values

# Size of the dementia group; after each shuffle the first n_dementia values are "dementia".
n_dementia = dementia_data.nWBV.count()

# OBSERVED statistic: the real difference in group means (healthy minus dementia).
observed_diff = ...  # hint: "-" is the minus operator
print("Observed difference in mean nWBV (healthy - dementia): %.4f" % observed_diff)

perm_diffs = np.zeros([n_iterations,])
for i in range(n_iterations): # loop to have use do the shuffling 10,000 times

    # Shuffle all pooled values WITHOUT replacement.
    shuffled = np.random.permutation(...)

    perm_dementia = shuffled[:n_dementia]   # first n_dementia values
    perm_healthy  = shuffled[n_dementia:]   # the rest


    perm_healthy_mean = perm_healthy.mean()
    perm_dementia_mean = perm_dementia.mean()

    # Save the difference in means for this current shuffled means (healthy mean - dementia mean).
    perm_diffs[i] = ...   


In [ ]:
# Plot the NULL DISTRIBUTION: the differences in means produced by shuffling the labels.
# (just run this cell to produce the figure) 
fig, ax = plt.subplots(1, figsize=(10, 8))
ax.hist(perm_diffs, rwidth=0.9, alpha=0.5, bins=30,
        label="Null distribution of (healthy - dementia) mean differences\n(from shuffling the labels)")
ax.axvline(observed_diff, color="C1", lw=3,
           label="Measured difference (healthy - dementia)")
ax.set_title("Permutation test: null distribution of (healthy - dementia) mean nWBV")
ax.set_xlabel("Difference in mean nWBV, healthy - dementia (A.U.)")
ax.set_ylabel("Count")
ax.legend()


### Step 4: P-value

The histogram is the **null distribution**: all (healthy - dementia) differences expected **if the labels didn't matter**. The emphasized line is the observed difference we **actually measured**.

The **p-value**: *if the null were true, how often would chance alone produce a difference at least as extreme as the one measured?* It is the fraction of shuffled differences as far out as, or farther than, the measured line.

- A **small** p-value: the measured difference is far in the tail — chance rarely produces it.
- A **large** p-value: the measured difference sits in the thick of the null — chance produces it often.

**This is a one-sided test.** because we expect dementia to be associated with *lower* nWBV, i.e. a *positive* (healthy - dementia) difference, so we count shuffles with difference `>=` the measured one.

In [ ]:
# Use the ">=" operator to find how many values in whole array of differences 
# calculate by the permutaiotn test are larger than or equal to observed_diff, 
# this will give a True/False (Boolean) for each shuffle that is counted using .sum() in the line below.

boolean_more_extreme = ...   # hint: us the  ">=" operator
count_more_extreme = boolean_more_extreme.sum() # counts the number of permuted differences that are equal or larger than the observed difference 

print("%d out of %d shuffled differences were as extreme or more than the measured difference."
      % (count_more_extreme, n_iterations))

# Calculate the fraction of extreme from the total number of shuffles to get the p-value.
p_value = ...                # hint: "/" is the division operator, and "n_iterations" is the number of total shuffles
print("p-value (uncorrected): %0.4f" % p_value)


#### A small correction to the p-value

If **none** of your shuffles were as extreme as the measured difference, the formula above gives exactly **0** — claiming chance could *never* produce your result. But your real data is itself one arrangement of the labels that produced exactly that difference, so it is not impossible.

The fix: count the real data as one more arrangement. Add 1 to the numerator and 1 to the denominator:

$$p = \frac{\text{count} + 1}{n_{\text{iterations}} + 1}$$

This is the recommended permutation-test p-value. **Just run the cell below.**

In [ ]:
# Corrected permutation-test p-value: count the real data as one more arrangement.
# (just run this cell to calculate the corrected p-value)
p_value_corrected = (count_more_extreme + 1) / (n_iterations + 1)
print("p-value (uncorrected): %0.4f" % p_value)
print("p-value (corrected):   %0.4f" % p_value_corrected)


Answer the questions in the Lab 1 assignment to report and interpret the information in this notebook.